# Phase 1 — Correct Temporal Duplicate Bug Retrieval Benchmark

This notebook replaces the earlier single-target evaluation with a leakage-safe,
multi-relevant benchmark for **BugHub Mozilla Firefox + Mozilla Core**.

Key rules:

1. Use the official `train.csv` and `test.csv` relevance sets.
2. Validation is the latest 15% of the official training period.
3. Test is the official chronological test split.
4. For a query at time `t`, candidates and relevant reports must satisfy
   `created_at < t`.
5. Test uses a rolling history: earlier test reports may be candidates for later
   test reports, but TF-IDF vocabulary/IDF is fitted only on official training data.
6. Report `Hit@K`, true multi-label `Recall@K`, `MRR@10`, and `nDCG@10`.

The default test baseline is chosen **before looking at test results**. Tune only
on validation, then set `SELECTED_TEST_BASELINE` once and run the final section.


In [ ]:
import gc
import html
import json
import math
import re
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 160)
RANDOM_SEED = 42


## 1. Paths and reproducibility configuration

The paths match the Google Drive folders shown in the screenshot. The local
fallback makes the same notebook executable outside Colab for smoke tests.

Set `MAX_EVAL_QUERIES = 100` for a quick smoke test. Use `None` for publishable
full results.


In [ ]:
COLAB_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/DuplicateDetection")
LOCAL_ROOT = Path(r"D:\Github\bughub")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

DATA_ROOT = COLAB_ROOT if COLAB_ROOT.exists() else LOCAL_ROOT
OUTPUT_ROOT = DATA_ROOT / "phase1_fixed_outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

PROJECTS = {
    "Firefox": {
        "main": DATA_ROOT / "Firefox" / "mozilla_firefox.csv",
        "train": DATA_ROOT / "Firefox" / "train.csv",
        "test": DATA_ROOT / "Firefox" / "test.csv",
    },
    "MozillaCore": {
        "main": DATA_ROOT / "MozillaCore" / "mozilla_core.csv",
        "train": DATA_ROOT / "MozillaCore" / "train.csv",
        "test": DATA_ROOT / "MozillaCore" / "test.csv",
    },
}

VALIDATION_FRACTION_OF_OFFICIAL_TRAIN = 0.15
TOP_KS = (1, 5, 10)
MAX_EVAL_QUERIES = None  # Use 100 only for a smoke test.
VALIDATION_BASELINES = ("word_tfidf", "hybrid_word_char")
SELECTED_TEST_BASELINE = "hybrid_word_char"  # Declare before running test.

for project, paths in PROJECTS.items():
    for kind, path in paths.items():
        assert path.exists(), f"Missing {project}/{kind}: {path}"

print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("MAX_EVAL_QUERIES:", MAX_EVAL_QUERIES)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATA_ROOT: /content/drive/MyDrive/Colab Notebooks/DuplicateDetection
OUTPUT_ROOT: /content/drive/MyDrive/Colab Notebooks/DuplicateDetection/phase1_fixed_outputs
MAX_EVAL_QUERIES: None


## 2. Load and normalize issues and official relevance files

`Duplicate` is a semicolon-separated multi-relevant set. It must not be reduced
to one target. Text normalization deliberately preserves stack symbols, error
codes, and identifiers; it only decodes HTML, replaces URLs, and normalizes
whitespace.


In [ ]:
def normalize_id(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    return re.sub(r"\.0$", "", text)


def normalize_text(value):
    if pd.isna(value):
        return ""
    text = html.unescape(str(value))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " <URL> ", text)
    return re.sub(r"\s+", " ", text).strip()


def parse_duplicate_set(value):
    if pd.isna(value):
        return frozenset()
    text = str(value).strip()
    if not text or text.upper() == "NULL":
        return frozenset()
    return frozenset(
        normalized
        for item in text.split(";")
        if (normalized := normalize_id(item))
    )


def load_project(project, paths):
    issues = pd.read_csv(paths["main"], low_memory=False)
    issues = issues.rename(columns={
        "Issue_id": "issue_id",
        "Priority": "priority",
        "Component": "component",
        "Duplicated_issue": "duplicate_of",
        "Title": "title",
        "Description": "description",
        "Resolution": "resolution",
        "Created_time": "created_at",
    })
    required = {
        "issue_id", "title", "description", "created_at",
        "duplicate_of", "component", "resolution",
    }
    assert required.issubset(issues.columns), required - set(issues.columns)

    issues["issue_id"] = issues["issue_id"].map(normalize_id)
    issues["created_at"] = pd.to_datetime(issues["created_at"], errors="coerce", utc=True)
    issues["title_clean"] = issues["title"].map(normalize_text)
    issues["description_clean"] = issues["description"].map(normalize_text)
    # Repeat title once to give it modest field weight in the word baseline.
    issues["text"] = (
        issues["title_clean"] + " . " + issues["title_clean"]
        + " . " + issues["description_clean"]
    ).str.strip()
    issues["project"] = project

    assert issues["issue_id"].notna().all()
    assert not issues["issue_id"].duplicated().any()
    assert issues["created_at"].notna().all()
    issues = issues.sort_values(["created_at", "issue_id"]).reset_index(drop=True)
    issues["position"] = np.arange(len(issues), dtype=np.int64)

    split_frames = {}
    for split in ("train", "test"):
        labels = pd.read_csv(paths[split], dtype=str)
        assert {"Issue_id", "Duplicate"}.issubset(labels.columns)
        labels = labels.rename(columns={"Issue_id": "issue_id", "Duplicate": "duplicate_raw"})
        labels["issue_id"] = labels["issue_id"].map(normalize_id)
        labels["relevant_ids"] = labels["duplicate_raw"].map(parse_duplicate_set)
        assert labels["issue_id"].notna().all()
        assert not labels["issue_id"].duplicated().any()
        split_frames[split] = labels[["issue_id", "relevant_ids"]].copy()

    all_ids = set(issues["issue_id"])
    train_ids = set(split_frames["train"]["issue_id"])
    test_ids = set(split_frames["test"]["issue_id"])
    assert train_ids.isdisjoint(test_ids), f"{project}: train/test ID overlap"
    assert train_ids | test_ids == all_ids, f"{project}: split does not cover main CSV"

    id_to_time = issues.set_index("issue_id")["created_at"].to_dict()
    id_to_text = issues.set_index("issue_id")["text"].to_dict()
    known = set(id_to_time)

    audit_rows = []
    for split, labels in split_frames.items():
        total_links = sum(len(x) for x in labels["relevant_ids"])
        missing = sum(
            target not in known
            for targets in labels["relevant_ids"]
            for target in targets
        )
        future = same = 0
        for query_id, targets in labels[["issue_id", "relevant_ids"]].itertuples(index=False):
            query_time = id_to_time[query_id]
            for target in targets:
                if target not in id_to_time:
                    continue
                future += id_to_time[target] > query_time
                same += id_to_time[target] == query_time
        audit_rows.append({
            "project": project,
            "split": split,
            "rows": len(labels),
            "labeled_queries": int((labels["relevant_ids"].map(len) > 0).sum()),
            "relevance_links": total_links,
            "missing_targets": missing,
            "future_targets": future,
            "same_time_targets": same,
        })

    return {
        "project": project,
        "issues": issues,
        "train_labels": split_frames["train"],
        "test_labels": split_frames["test"],
        "id_to_time": id_to_time,
        "id_to_text": id_to_text,
        "audit": pd.DataFrame(audit_rows),
    }


datasets = {
    project: load_project(project, paths)
    for project, paths in PROJECTS.items()
}

data_audit = pd.concat([data["audit"] for data in datasets.values()], ignore_index=True)
display(data_audit)


,project,split,rows,labeled_queries,relevance_links,missing_targets,future_targets,same_time_targets
0,Firefox,train,92651,30026,310378,0,155224,22
1,Firefox,test,23163,4236,109926,0,0,2
2,MozillaCore,train,164055,50122,281511,0,140798,144
3,MozillaCore,test,41014,4115,8937,0,0,1


## 3. Data-gate assertions

The official test files are suitable for rolling temporal retrieval only when
all labeled targets exist and none is from the future. Training label files may
contain future cluster members; validation construction filters those targets
at each query timestamp.


In [ ]:
dataset_summary = []
for project, data in datasets.items():
    issues = data["issues"]
    test_audit = data["audit"].query("split == 'test'").iloc[0]
    assert test_audit["missing_targets"] == 0
    assert test_audit["future_targets"] == 0
    assert not issues["issue_id"].duplicated().any()
    dataset_summary.append({
        "project": project,
        "issues": len(issues),
        "empty_text": int(issues["text"].str.len().eq(0).sum()),
        "created_min": issues["created_at"].min(),
        "created_max": issues["created_at"].max(),
        "official_train": len(data["train_labels"]),
        "official_test": len(data["test_labels"]),
        "test_labeled_queries": int(test_audit["labeled_queries"]),
        "test_relevance_links": int(test_audit["relevance_links"]),
    })

dataset_summary = pd.DataFrame(dataset_summary)
display(dataset_summary)
print("DATA GATE: PASS")


,project,issues,empty_text,created_min,created_max,official_train,official_test,test_labeled_queries,test_relevance_links
0,Firefox,115814,0,1999-07-30 22:55:51+00:00,2014-01-01 06:10:17+00:00,92651,23163,4236,109926
1,MozillaCore,205069,0,1997-03-28 16:00:00+00:00,2013-12-31 20:02:55+00:00,164055,41014,4115,8937


DATA GATE: PASS


## 4. Construct validation and test query sets

For every query, relevant reports are filtered to IDs that:

- exist in the same project corpus;
- have usable text;
- were created strictly before the query.

Queries with no remaining historical relevant report are not evaluable and are
reported separately. This prevents future-target leakage.


In [ ]:
def filter_temporal_queries(data, labels, allowed_query_ids=None):
    id_to_time = data["id_to_time"]
    id_to_text = data["id_to_text"]
    records = []
    audit = {
        "labeled_queries": 0,
        "empty_query_text": 0,
        "no_historical_relevant": 0,
        "future_or_same_relevant_removed": 0,
        "missing_or_empty_relevant_removed": 0,
    }
    allowed_query_ids = None if allowed_query_ids is None else set(allowed_query_ids)

    for query_id, raw_relevant in labels[["issue_id", "relevant_ids"]].itertuples(index=False):
        if allowed_query_ids is not None and query_id not in allowed_query_ids:
            continue
        if not raw_relevant:
            continue
        audit["labeled_queries"] += 1
        if not id_to_text.get(query_id, ""):
            audit["empty_query_text"] += 1
            continue

        query_time = id_to_time[query_id]
        relevant = set()
        for target in raw_relevant:
            if target not in id_to_time or not id_to_text.get(target, ""):
                audit["missing_or_empty_relevant_removed"] += 1
                continue
            if id_to_time[target] >= query_time:
                audit["future_or_same_relevant_removed"] += 1
                continue
            relevant.add(target)

        if not relevant:
            audit["no_historical_relevant"] += 1
            continue
        records.append({
            "query_id": query_id,
            "query_time": query_time,
            "relevant_ids": frozenset(relevant),
            "num_relevant": len(relevant),
        })

    query_frame = pd.DataFrame(records).sort_values(["query_time", "query_id"]).reset_index(drop=True)
    audit["evaluable_queries"] = len(query_frame)
    audit["evaluable_relevance_links"] = int(query_frame["num_relevant"].sum()) if len(query_frame) else 0
    return query_frame, audit


query_sets = {}
query_audits = []
for project, data in datasets.items():
    issues = data["issues"]
    train_labels = data["train_labels"]
    official_train_ids = set(train_labels["issue_id"])
    train_issues = issues[issues["issue_id"].isin(official_train_ids)].sort_values(["created_at", "issue_id"])

    validation_size = math.ceil(len(train_issues) * VALIDATION_FRACTION_OF_OFFICIAL_TRAIN)
    validation_ids = set(train_issues.tail(validation_size)["issue_id"])
    dev_fit_ids = set(train_issues.iloc[:-validation_size]["issue_id"])

    validation_queries, validation_audit = filter_temporal_queries(
        data, train_labels, validation_ids
    )
    test_queries, test_audit = filter_temporal_queries(data, data["test_labels"])

    assert all(
        data["id_to_time"][target] < row.query_time
        for row in validation_queries.itertuples()
        for target in row.relevant_ids
    )
    assert all(
        data["id_to_time"][target] < row.query_time
        for row in test_queries.itertuples()
        for target in row.relevant_ids
    )

    query_sets[project] = {
        "dev_fit_ids": dev_fit_ids,
        "official_train_ids": official_train_ids,
        "validation": validation_queries,
        "test": test_queries,
    }
    for split, audit in (("validation", validation_audit), ("test", test_audit)):
        query_audits.append({"project": project, "split": split, **audit})

query_audits = pd.DataFrame(query_audits)
display(query_audits)
print("TEMPORAL GROUND TRUTH: PASS")


,project,split,labeled_queries,empty_query_text,no_historical_relevant,future_or_same_relevant_removed,missing_or_empty_relevant_removed,evaluable_queries,evaluable_relevance_links
0,Firefox,validation,3986,0,926,9131,0,3060,17577
1,Firefox,test,4236,0,0,2,0,4236,109924
2,MozillaCore,validation,5071,0,1714,4636,0,3357,7524
3,MozillaCore,test,4115,0,0,1,0,4115,8936


TEMPORAL GROUND TRUTH: PASS


## 5. Metrics and rolling retrieval engine

`Hit@K` checks whether at least one relevant item is found. `Recall@K` measures
the fraction of all historical relevant items retrieved. `MRR@10` is truncated
at rank 10 and `nDCG@10` uses binary relevance.


In [ ]:
def ranking_metrics(retrieved_ids, relevant_ids, ks=TOP_KS):
    relevant = set(relevant_ids)
    result = {}
    for k in ks:
        hits = len(set(retrieved_ids[:k]) & relevant)
        result[f"hit@{k}"] = float(hits > 0)
        result[f"recall@{k}"] = hits / len(relevant)

    first_rank = next(
        (rank for rank, issue_id in enumerate(retrieved_ids, start=1) if issue_id in relevant),
        None,
    )
    result["mrr@10"] = 0.0 if first_rank is None else 1.0 / first_rank

    dcg = sum(
        1.0 / math.log2(rank + 1)
        for rank, issue_id in enumerate(retrieved_ids[:10], start=1)
        if issue_id in relevant
    )
    ideal_hits = min(len(relevant), 10)
    idcg = sum(1.0 / math.log2(rank + 1) for rank in range(1, ideal_hits + 1))
    result["ndcg@10"] = dcg / idcg if idcg else 0.0
    return result


def top_k_indices(scores, k):
    k = min(k, len(scores))
    if k == 0:
        return np.array([], dtype=np.int64)
    if k == len(scores):
        candidates = np.arange(len(scores))
    else:
        candidates = np.argpartition(-scores, k - 1)[:k]
    # Deterministic: score descending, then corpus position ascending.
    order = np.lexsort((candidates, -scores[candidates]))
    return candidates[order]


def fit_feature_matrix(issues, fit_ids, baseline):
    fit_mask = issues["issue_id"].isin(fit_ids)
    fit_rows = issues.loc[fit_mask]
    assert len(fit_rows) > 0

    word_vectorizer = TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.995,
        max_features=200_000,
        sublinear_tf=True,
        dtype=np.float32,
    )
    word_vectorizer.fit(fit_rows["text"])
    word_matrix = normalize(word_vectorizer.transform(issues["text"]), copy=False)

    metadata = {
        "word_vocabulary": len(word_vectorizer.vocabulary_),
        "char_vocabulary": 0,
    }
    if baseline == "word_tfidf":
        return word_matrix.tocsr(), metadata
    if baseline != "hybrid_word_char":
        raise ValueError(f"Unknown baseline: {baseline}")

    # Character n-grams on title are useful for stack symbols, identifiers,
    # version strings, typos, and error codes while keeping memory controlled.
    char_vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        lowercase=True,
        ngram_range=(3, 5),
        min_df=3,
        max_features=100_000,
        sublinear_tf=True,
        dtype=np.float32,
    )
    char_vectorizer.fit(fit_rows["title_clean"])
    char_matrix = normalize(char_vectorizer.transform(issues["title_clean"]), copy=False)
    combined = hstack(
        [word_matrix * np.float32(math.sqrt(0.80)),
         char_matrix * np.float32(math.sqrt(0.20))],
        format="csr",
        dtype=np.float32,
    )
    combined = normalize(combined, copy=False)
    metadata["char_vocabulary"] = len(char_vectorizer.vocabulary_)
    del char_matrix, word_matrix
    gc.collect()
    return combined, metadata


def evaluate_rolling(data, queries, fit_ids, baseline, max_queries=None):
    issues = data["issues"]
    matrix, feature_metadata = fit_feature_matrix(issues, fit_ids, baseline)
    issue_ids = issues["issue_id"].to_numpy()
    created_ns = issues["created_at"].astype("int64").to_numpy()
    id_to_position = dict(zip(issue_ids, issues["position"]))

    eval_queries = queries
    if max_queries is not None and len(eval_queries) > max_queries:
        # Deterministic chronological sample spread across the full period.
        selected = np.linspace(0, len(eval_queries) - 1, max_queries, dtype=int)
        eval_queries = eval_queries.iloc[selected].drop_duplicates("query_id")

    rows = []
    started = time.time()
    for number, row in enumerate(eval_queries.itertuples(index=False), start=1):
        query_pos = id_to_position[row.query_id]
        query_time_ns = int(row.query_time.value)
        candidate_end = int(np.searchsorted(created_ns, query_time_ns, side="left"))
        assert candidate_end <= query_pos or created_ns[query_pos] == query_time_ns
        assert candidate_end > 0

        scores = (matrix[:candidate_end] @ matrix[query_pos].T).toarray().ravel()
        local_top = top_k_indices(scores, max(TOP_KS))
        retrieved = issue_ids[local_top].tolist()
        assert row.query_id not in retrieved
        assert all(data["id_to_time"][item] < row.query_time for item in retrieved)

        metrics = ranking_metrics(retrieved, row.relevant_ids)
        rows.append({
            "project": data["project"],
            "baseline": baseline,
            "query_id": row.query_id,
            "query_time": row.query_time,
            "num_candidates": candidate_end,
            "num_relevant": len(row.relevant_ids),
            "retrieved_ids": ";".join(retrieved),
            **metrics,
        })
        if number % 500 == 0:
            print(f"  {data['project']} / {baseline}: {number}/{len(eval_queries)}")

    elapsed = time.time() - started
    results = pd.DataFrame(rows)
    metric_columns = [
        *(f"hit@{k}" for k in TOP_KS),
        *(f"recall@{k}" for k in TOP_KS),
        "mrr@10", "ndcg@10",
    ]
    summary = {
        "project": data["project"],
        "baseline": baseline,
        "queries": len(results),
        "median_candidates": int(results["num_candidates"].median()) if len(results) else 0,
        "seconds": round(elapsed, 2),
        **feature_metadata,
        **{column: results[column].mean() for column in metric_columns},
    }
    del matrix
    gc.collect()
    return results, summary


# Metric unit tests: one of two relevant documents at rank 2.
metric_check = ranking_metrics(["x", "b", "y"], {"a", "b"}, ks=(1, 2, 3))
assert metric_check["hit@1"] == 0
assert metric_check["hit@2"] == 1
assert metric_check["recall@2"] == 0.5
assert metric_check["mrr@10"] == 0.5
print("METRIC TESTS: PASS")


METRIC TESTS: PASS


## 6. Validation — choose the baseline here, not on test

This section fits only on the early official-training history and evaluates on
the latest 15% of official training reports with rolling historical candidates.
If runtime is an issue, temporarily set `MAX_EVAL_QUERIES = 100`; do not publish
sampled scores as full benchmark results.


In [ ]:
validation_summaries = []
validation_results = {}

for project, data in datasets.items():
    print(f"\n=== VALIDATION: {project} ===")
    for baseline in VALIDATION_BASELINES:
        results, summary = evaluate_rolling(
            data=data,
            queries=query_sets[project]["validation"],
            fit_ids=query_sets[project]["dev_fit_ids"],
            baseline=baseline,
            max_queries=MAX_EVAL_QUERIES,
        )
        validation_results[(project, baseline)] = results
        validation_summaries.append(summary)
        display(pd.DataFrame([summary]))

validation_summary = pd.DataFrame(validation_summaries)
display(validation_summary.sort_values(["project", "hit@10"], ascending=[True, False]))
validation_summary.to_csv(OUTPUT_ROOT / "validation_summary.csv", index=False)



=== VALIDATION: Firefox ===
  Firefox / word_tfidf: 500/3060
  Firefox / word_tfidf: 1000/3060
  Firefox / word_tfidf: 1500/3060
  Firefox / word_tfidf: 2000/3060
  Firefox / word_tfidf: 2500/3060
  Firefox / word_tfidf: 3000/3060


,project,baseline,queries,median_candidates,seconds,word_vocabulary,char_vocabulary,hit@1,hit@5,hit@10,recall@1,recall@5,recall@10,mrr@10,ndcg@10
0,Firefox,word_tfidf,3060,86041,922.28,200000,0,0.227124,0.413072,0.492484,0.094967,0.214633,0.275922,0.304705,0.23713


  Firefox / hybrid_word_char: 500/3060
  Firefox / hybrid_word_char: 1000/3060
  Firefox / hybrid_word_char: 1500/3060
  Firefox / hybrid_word_char: 2000/3060
  Firefox / hybrid_word_char: 2500/3060
  Firefox / hybrid_word_char: 3000/3060


,project,baseline,queries,median_candidates,seconds,word_vocabulary,char_vocabulary,hit@1,hit@5,hit@10,recall@1,recall@5,recall@10,mrr@10,ndcg@10
0,Firefox,hybrid_word_char,3060,86041,1462.74,200000,75623,0.253922,0.447059,0.533007,0.112381,0.232403,0.303635,0.337554,0.264077



=== VALIDATION: MozillaCore ===
  MozillaCore / word_tfidf: 500/3357
  MozillaCore / word_tfidf: 1000/3357
  MozillaCore / word_tfidf: 1500/3357
  MozillaCore / word_tfidf: 2000/3357
  MozillaCore / word_tfidf: 2500/3357
  MozillaCore / word_tfidf: 3000/3357


,project,baseline,queries,median_candidates,seconds,word_vocabulary,char_vocabulary,hit@1,hit@5,hit@10,recall@1,recall@5,recall@10,mrr@10,ndcg@10
0,MozillaCore,word_tfidf,3357,151724,1207.37,200000,0,0.184987,0.319333,0.371463,0.130955,0.246705,0.292455,0.2406,0.230388


  MozillaCore / hybrid_word_char: 500/3357
  MozillaCore / hybrid_word_char: 1000/3357
  MozillaCore / hybrid_word_char: 1500/3357
  MozillaCore / hybrid_word_char: 2000/3357
  MozillaCore / hybrid_word_char: 2500/3357
  MozillaCore / hybrid_word_char: 3000/3357


,project,baseline,queries,median_candidates,seconds,word_vocabulary,char_vocabulary,hit@1,hit@5,hit@10,recall@1,recall@5,recall@10,mrr@10,ndcg@10
0,MozillaCore,hybrid_word_char,3357,151724,2214.0,200000,100000,0.220435,0.373548,0.435806,0.159147,0.291835,0.349618,0.285282,0.27518


,project,baseline,queries,median_candidates,seconds,word_vocabulary,char_vocabulary,hit@1,hit@5,hit@10,recall@1,recall@5,recall@10,mrr@10,ndcg@10
1,Firefox,hybrid_word_char,3060,86041,1462.74,200000,75623,0.253922,0.447059,0.533007,0.112381,0.232403,0.303635,0.337554,0.264077
0,Firefox,word_tfidf,3060,86041,922.28,200000,0,0.227124,0.413072,0.492484,0.094967,0.214633,0.275922,0.304705,0.237130
3,MozillaCore,hybrid_word_char,3357,151724,2214.00,200000,100000,0.220435,0.373548,0.435806,0.159147,0.291835,0.349618,0.285282,0.275180
2,MozillaCore,word_tfidf,3357,151724,1207.37,200000,0,0.184987,0.319333,0.371463,0.130955,0.246705,0.292455,0.240600,0.230388


## 7. Final official test — run once after baseline selection

The vectorizer is fitted on the complete official training split. Earlier test
reports may enter the rolling candidate index for later queries, but no test text
affects vocabulary or IDF fitting.


In [ ]:
assert SELECTED_TEST_BASELINE in VALIDATION_BASELINES
test_summaries = []
test_results = {}

for project, data in datasets.items():
    print(f"\n=== TEST: {project} / {SELECTED_TEST_BASELINE} ===")
    results, summary = evaluate_rolling(
        data=data,
        queries=query_sets[project]["test"],
        fit_ids=query_sets[project]["official_train_ids"],
        baseline=SELECTED_TEST_BASELINE,
        max_queries=MAX_EVAL_QUERIES,
    )
    test_results[project] = results
    test_summaries.append(summary)
    results.to_csv(OUTPUT_ROOT / f"{project.lower()}_test_query_results.csv", index=False)

test_summary = pd.DataFrame(test_summaries)
display(test_summary)

metric_columns = [
    *(f"hit@{k}" for k in TOP_KS),
    *(f"recall@{k}" for k in TOP_KS),
    "mrr@10", "ndcg@10",
]
macro = {
    "project": "MACRO_PROJECT_AVERAGE",
    "baseline": SELECTED_TEST_BASELINE,
    "queries": int(test_summary["queries"].sum()),
    **{column: test_summary[column].mean() for column in metric_columns},
}
test_summary_with_macro = pd.concat(
    [test_summary, pd.DataFrame([macro])], ignore_index=True
)
display(test_summary_with_macro)
test_summary_with_macro.to_csv(OUTPUT_ROOT / "test_summary.csv", index=False)



=== TEST: Firefox / hybrid_word_char ===
  Firefox / hybrid_word_char: 500/4236
  Firefox / hybrid_word_char: 1000/4236
  Firefox / hybrid_word_char: 1500/4236
  Firefox / hybrid_word_char: 2000/4236
  Firefox / hybrid_word_char: 2500/4236
  Firefox / hybrid_word_char: 3000/4236
  Firefox / hybrid_word_char: 3500/4236
  Firefox / hybrid_word_char: 4000/4236

=== TEST: MozillaCore / hybrid_word_char ===
  MozillaCore / hybrid_word_char: 500/4115
  MozillaCore / hybrid_word_char: 1000/4115
  MozillaCore / hybrid_word_char: 1500/4115
  MozillaCore / hybrid_word_char: 2000/4115
  MozillaCore / hybrid_word_char: 2500/4115
  MozillaCore / hybrid_word_char: 3000/4115
  MozillaCore / hybrid_word_char: 3500/4115
  MozillaCore / hybrid_word_char: 4000/4115


,project,baseline,queries,median_candidates,seconds,word_vocabulary,char_vocabulary,hit@1,hit@5,hit@10,recall@1,recall@5,recall@10,mrr@10,ndcg@10
0,Firefox,hybrid_word_char,4236,103707,2265.38,200000,83498,0.258026,0.439330,0.516525,0.128808,0.244776,0.300384,0.334694,0.278152
1,MozillaCore,hybrid_word_char,4115,183632,3218.34,200000,100000,0.215553,0.351397,0.409964,0.163551,0.284649,0.342338,0.273452,0.269617


,project,baseline,queries,median_candidates,seconds,word_vocabulary,char_vocabulary,hit@1,hit@5,hit@10,recall@1,recall@5,recall@10,mrr@10,ndcg@10
0,Firefox,hybrid_word_char,4236,103707.0,2265.38,200000.0,83498.0,0.258026,0.439330,0.516525,0.128808,0.244776,0.300384,0.334694,0.278152
1,MozillaCore,hybrid_word_char,4115,183632.0,3218.34,200000.0,100000.0,0.215553,0.351397,0.409964,0.163551,0.284649,0.342338,0.273452,0.269617
2,MACRO_PROJECT_AVERAGE,hybrid_word_char,8351,NaN,NaN,NaN,NaN,0.236790,0.395363,0.463244,0.146179,0.264712,0.321361,0.304073,0.273885


## 8. Final leakage checks and manifest

The manifest records whether the run was full or sampled. Only results with
`MAX_EVAL_QUERIES = None` should be reported as the final benchmark.


In [ ]:
for project, results in test_results.items():
    data = datasets[project]
    assert results["query_id"].is_unique
    for row in results.itertuples(index=False):
        query_time = data["id_to_time"][row.query_id]
        assert all(
            data["id_to_time"][retrieved_id] < query_time
            for retrieved_id in row.retrieved_ids.split(";")
            if retrieved_id
        )

manifest = {
    "protocol": "official per-project split; rolling temporal candidate history",
    "selected_test_baseline": SELECTED_TEST_BASELINE,
    "top_ks": list(TOP_KS),
    "max_eval_queries": MAX_EVAL_QUERIES,
    "full_publishable_run": MAX_EVAL_QUERIES is None,
    "random_seed": RANDOM_SEED,
    "data_root": str(DATA_ROOT),
    "projects": {
        project: {
            "issues": len(data["issues"]),
            "official_train": len(data["train_labels"]),
            "official_test": len(data["test_labels"]),
            "evaluable_test_queries": len(query_sets[project]["test"]),
        }
        for project, data in datasets.items()
    },
}

(OUTPUT_ROOT / "run_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)
print(json.dumps(manifest, indent=2))
print("\nLEAKAGE CHECKS: PASS")
print("Artifacts:", OUTPUT_ROOT)


{
  "protocol": "official per-project split; rolling temporal candidate history",
  "selected_test_baseline": "hybrid_word_char",
  "top_ks": [
    1,
    5,
    10
  ],
  "max_eval_queries": null,
  "full_publishable_run": true,
  "random_seed": 42,
  "data_root": "/content/drive/MyDrive/Colab Notebooks/DuplicateDetection",
  "projects": {
    "Firefox": {
      "issues": 115814,
      "official_train": 92651,
      "official_test": 23163,
      "evaluable_test_queries": 4236
    },
    "MozillaCore": {
      "issues": 205069,
      "official_train": 164055,
      "official_test": 41014,
      "evaluable_test_queries": 4115
    }
  }
}

LEAKAGE CHECKS: PASS
Artifacts: /content/drive/MyDrive/Colab Notebooks/DuplicateDetection/phase1_fixed_outputs


## Interpretation

- `Hit@10` answers: *Did the Top-10 contain at least one known duplicate?*
- `Recall@10` answers: *What fraction of all known historical duplicates were
  returned in the Top-10?*
- `MRR@10` rewards putting the first relevant report near the top.
- `nDCG@10` rewards retrieving multiple relevant reports at high ranks.

Because duplicate labels can still be incomplete, an unlabelled retrieved report
is not guaranteed to be a true negative. For the competition report, manually
audit a reproducible random sample of apparent false positives.
